# E-Commerce Exploratory Data Analysis

**Dataset:** Online Retail Transactions  
**Goal:** Understand data quality, customer behavior, product performance, and revenue trends to inform downstream modeling and business decisions.

---

## 1. Setup & Imports

Load all libraries, configure plot aesthetics, and set pandas display options for comfortable viewing throughout the notebook.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Plot style ──────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.figsize': (10, 5),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

# ── Pandas display ───────────────────────────────────────────────────────────
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_colwidth', 40)

print('Libraries loaded successfully.')

---
## 2. Data Loading

Load the CSV, preview the first rows, and inspect the schema. We also derive two important columns right away — `Revenue` (Quantity × UnitPrice) and a parsed `InvoiceDate` — so downstream cells can use them immediately.

In [ ]:
# ── Locate the CSV relative to this notebook ────────────────────────────────
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
DATA_PATH = os.path.join(NOTEBOOK_DIR, 'ecommerce_analysis', 'data', 'raw', 'data.csv')

df_raw = pd.read_csv(DATA_PATH, encoding='ISO-8859-1', low_memory=False)
print(f'Loaded  {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')

In [ ]:
# Preview
display(df_raw.head(8))

In [ ]:
# Schema overview
schema = pd.DataFrame({
    'dtype': df_raw.dtypes,
    'non_null': df_raw.count(),
    'null_count': df_raw.isna().sum(),
    'sample_value': df_raw.iloc[0]
})
display(schema)

In [ ]:
# ── Derive Revenue and parse InvoiceDate ─────────────────────────────────────
df = df_raw.copy()

# Parse date column if present
DATE_COL = None
for col in df.columns:
    if 'date' in col.lower() or 'time' in col.lower():
        try:
            df[col] = pd.to_datetime(df[col], infer_datetime_format=True)
            DATE_COL = col
            print(f'Parsed date column: "{DATE_COL}"  range: {df[DATE_COL].min()} → {df[DATE_COL].max()}')
            break
        except Exception:
            pass

# Revenue column
num_cols = df.select_dtypes(include='number').columns.tolist()
qty_col = next((c for c in num_cols if 'qty' in c.lower() or 'quant' in c.lower()), None)
price_col = next((c for c in num_cols if 'price' in c.lower() or 'unit' in c.lower()), None)

if qty_col and price_col:
    df['Revenue'] = df[qty_col] * df[price_col]
    print(f'Revenue = {qty_col} × {price_col}  |  Total: £{df["Revenue"].sum():,.0f}')
else:
    print('Could not auto-detect Quantity/Price columns — Revenue not created.')

# Cache column lists for reuse
NUMERIC_COLS = df.select_dtypes(include='number').columns.tolist()
CAT_COLS     = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'\nNumeric columns : {NUMERIC_COLS}')
print(f'Categorical cols: {CAT_COLS}')

---
## 3. Data Quality Assessment

Before drawing conclusions we must understand the completeness and cleanliness of the data. We check missing values, duplicates, and cardinality, and review summary statistics for anomalies (e.g. negative quantities from returns).

In [ ]:
# ── Missing values ────────────────────────────────────────────────────────────
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct':   (df.isna().mean() * 100).round(2)
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

if missing.empty:
    print('No missing values found.')
else:
    display(missing)
    fig, ax = plt.subplots(figsize=(8, max(3, len(missing) * 0.5)))
    missing['missing_pct'].plot(kind='barh', ax=ax, color='salmon', edgecolor='white')
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Value Rate by Column')
    for p in ax.patches:
        ax.text(p.get_width() + 0.3, p.get_y() + p.get_height()/2,
                f'{p.get_width():.1f}%', va='center', fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Duplicate rows ────────────────────────────────────────────────────────────
n_dup = df.duplicated().sum()
print(f'Duplicate rows: {n_dup:,}  ({n_dup/len(df)*100:.2f}% of dataset)')
if n_dup > 0:
    display(df[df.duplicated(keep=False)].head(4))

In [ ]:
# ── Cardinality of categorical columns ────────────────────────────────────────
cardinality = pd.DataFrame({
    'unique_values': df[CAT_COLS].nunique(),
    'pct_unique':    (df[CAT_COLS].nunique() / len(df) * 100).round(2)
}).sort_values('unique_values', ascending=False)
display(cardinality)

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────────
display(df[NUMERIC_COLS].describe().T)

In [ ]:
# ── Flag anomalies: negative quantities (returns/cancellations) ───────────────
if qty_col:
    neg_qty = df[df[qty_col] < 0]
    print(f'Rows with negative {qty_col}: {len(neg_qty):,}  ({len(neg_qty)/len(df)*100:.1f}%)')
    print(f'These likely represent returns/cancellations and should be handled before modeling.')

if price_col:
    zero_price = df[df[price_col] == 0]
    print(f'\nRows with {price_col} == 0: {len(zero_price):,}')

---
## 4. Univariate Analysis

Examine each variable individually. For numerical columns we use histograms with KDE overlays to see distributions; for categoricals we show the top-N value frequencies. Boxplots reveal outliers using the IQR fence method.

In [ ]:
# ── Histograms + KDE for all numeric columns ──────────────────────────────────
plot_num_cols = [c for c in NUMERIC_COLS if df[c].nunique() > 5]

n_cols = 2
n_rows = (len(plot_num_cols) + 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes.flatten()

for i, col in enumerate(plot_num_cols):
    data = df[col].dropna()
    # Clip extreme outliers for readability (keep 1st–99th pct)
    p1, p99 = data.quantile(0.01), data.quantile(0.99)
    data_clipped = data.clip(p1, p99)
    axes[i].hist(data_clipped, bins=60, color='steelblue', edgecolor='white', alpha=0.7, density=True)
    data_clipped.plot.kde(ax=axes[i], color='tomato', linewidth=2)
    axes[i].set_title(f'Distribution of {col}\n(clipped to 1st–99th pct)')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Density')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Feature Distributions', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Bar charts for categorical columns (top 15 values each) ───────────────────
TOP_N = 15

# Skip ID-like or near-unique columns
plot_cat_cols = [c for c in CAT_COLS if df[c].nunique() <= 200]

for col in plot_cat_cols:
    counts = df[col].value_counts().head(TOP_N)
    fig, ax = plt.subplots(figsize=(10, 4))
    counts.plot(kind='bar', ax=ax, color=sns.color_palette('muted', len(counts)), edgecolor='white')
    ax.set_title(f'Top {TOP_N} Values — {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=35)
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2, p.get_height()),
                    ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Outlier detection: boxplots + IQR summary ─────────────────────────────────
n_cols_box = 2
n_rows_box = (len(plot_num_cols) + 1) // n_cols_box
fig, axes = plt.subplots(n_rows_box, n_cols_box, figsize=(14, 4 * n_rows_box))
axes = axes.flatten()

outlier_summary = []
for i, col in enumerate(plot_num_cols):
    data = df[col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((data < lower) | (data > upper)).sum()
    outlier_summary.append({'column': col, 'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
                             'lower_fence': lower, 'upper_fence': upper,
                             'outlier_count': n_out, 'outlier_pct': round(n_out/len(data)*100, 2)})

    axes[i].boxplot(data.clip(data.quantile(0.001), data.quantile(0.999)),
                    vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightblue'),
                    medianprops=dict(color='tomato', linewidth=2))
    axes[i].set_title(f'{col}\n(IQR outliers: {n_out:,} = {n_out/len(data)*100:.1f}%)')
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Outlier Detection via Boxplots (1‰–99.9‰ clipped)', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

display(pd.DataFrame(outlier_summary).set_index('column'))

---
## 5. Bivariate & Multivariate Analysis

Now we look at relationships between variables — which features move together, which products drive the most revenue, how customers differ in spend, and what the order-value distribution looks like.

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
corr = df[NUMERIC_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(max(6, len(NUMERIC_COLS)), max(5, len(NUMERIC_COLS) - 1)))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix (lower triangle)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter plots for top correlated pairs ────────────────────────────────────
corr_pairs = (
    corr.where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
    .stack()
    .abs()
    .sort_values(ascending=False)
)
top_pairs = corr_pairs[corr_pairs > 0.3].head(4).index.tolist()

if top_pairs:
    fig, axes = plt.subplots(1, len(top_pairs), figsize=(5 * len(top_pairs), 5))
    if len(top_pairs) == 1:
        axes = [axes]
    for ax, (c1, c2) in zip(axes, top_pairs):
        sample = df[[c1, c2]].dropna().sample(min(3000, len(df)), random_state=42)
        ax.scatter(sample[c1], sample[c2], alpha=0.3, s=10, color='steelblue')
        ax.set_xlabel(c1)
        ax.set_ylabel(c2)
        r = corr.loc[c1, c2]
        ax.set_title(f'{c1} vs {c2}\nr = {r:.2f}')
    plt.suptitle('Scatter Plots — Highly Correlated Pairs', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No pairs with |r| > 0.3 found.')

In [ ]:
# ── Top products by Revenue ────────────────────────────────────────────────────
desc_col  = next((c for c in CAT_COLS if 'desc' in c.lower()), None)
rev_col   = 'Revenue' if 'Revenue' in df.columns else None

if desc_col and rev_col:
    top_products = (
        df.groupby(desc_col)[rev_col].sum()
          .sort_values(ascending=False)
          .head(15)
    )
    fig, ax = plt.subplots(figsize=(11, 5))
    top_products.plot(kind='bar', ax=ax, color=sns.color_palette('Blues_r', 15), edgecolor='white')
    ax.set_title('Top 15 Products by Total Revenue')
    ax.set_xlabel('Product Description')
    ax.set_ylabel('Revenue (£)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('Description or Revenue column not found — skipping top-products chart.')

In [ ]:
# ── Revenue by Country (top 10) ───────────────────────────────────────────────
country_col = next((c for c in CAT_COLS if 'country' in c.lower()), None)

if country_col and rev_col:
    rev_by_country = (
        df.groupby(country_col)[rev_col].sum()
          .sort_values(ascending=False)
          .head(10)
    )
    fig, ax = plt.subplots(figsize=(10, 4))
    rev_by_country.plot(kind='bar', ax=ax, color=sns.color_palette('viridis', 10), edgecolor='white')
    ax.set_title('Top 10 Countries by Revenue')
    ax.set_xlabel('Country')
    ax.set_ylabel('Revenue (£)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e6:.1f}M'))
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Average Order Value (AOV) distribution ────────────────────────────────────
invoice_col = next((c for c in df.columns if 'invoice' in c.lower() and 'no' in c.lower()
                    or 'invoice' in c.lower() and 'id' in c.lower()), None)
if invoice_col is None:
    invoice_col = next((c for c in df.columns if 'invoice' in c.lower()), None)

if invoice_col and rev_col:
    order_value = df.groupby(invoice_col)[rev_col].sum()
    order_value = order_value[order_value > 0]  # exclude returns

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Full distribution (clipped)
    clip_val = order_value.quantile(0.98)
    axes[0].hist(order_value.clip(upper=clip_val), bins=80, color='teal', edgecolor='white', alpha=0.8)
    axes[0].axvline(order_value.mean(), color='tomato', lw=2, label=f'Mean = £{order_value.mean():.0f}')
    axes[0].axvline(order_value.median(), color='gold', lw=2, linestyle='--', label=f'Median = £{order_value.median():.0f}')
    axes[0].set_title('Order Value Distribution (clipped at 98th pct)')
    axes[0].set_xlabel('Order Value (£)')
    axes[0].set_ylabel('Number of Orders')
    axes[0].legend()

    # Log scale for full range
    axes[1].hist(np.log1p(order_value), bins=80, color='slateblue', edgecolor='white', alpha=0.8)
    axes[1].set_title('Order Value Distribution (log₁₊ₓ scale)')
    axes[1].set_xlabel('log(1 + Order Value)')
    axes[1].set_ylabel('Number of Orders')

    plt.suptitle('Average Order Value Analysis', fontsize=14)
    plt.tight_layout()
    plt.show()

    print(f'Orders analysed : {len(order_value):,}')
    print(f'Mean order value: £{order_value.mean():.2f}')
    print(f'Median          : £{order_value.median():.2f}')
    print(f'Std dev         : £{order_value.std():.2f}')

In [ ]:
# ── Customer purchase frequency ───────────────────────────────────────────────
cust_col = next((c for c in df.columns if 'customer' in c.lower() or 'cust' in c.lower()), None)

if cust_col and invoice_col:
    # Filter valid customers (non-null ID)
    cust_df = df.dropna(subset=[cust_col])
    purchase_freq = cust_df.groupby(cust_col)[invoice_col].nunique().rename('orders')

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Histogram
    cap = purchase_freq.quantile(0.95)
    axes[0].hist(purchase_freq.clip(upper=cap), bins=50, color='mediumseagreen', edgecolor='white')
    axes[0].set_title('Customer Purchase Frequency\n(capped at 95th pct)')
    axes[0].set_xlabel('Number of Orders')
    axes[0].set_ylabel('Number of Customers')

    # Segment breakdown
    bins   = [0, 1, 3, 10, np.inf]
    labels = ['1 order', '2–3 orders', '4–10 orders', '10+ orders']
    seg    = pd.cut(purchase_freq, bins=bins, labels=labels).value_counts().sort_index()
    seg.plot(kind='bar', ax=axes[1], color=sns.color_palette('Set2', 4), edgecolor='white')
    axes[1].set_title('Customer Segments by Purchase Count')
    axes[1].set_xlabel('Segment')
    axes[1].set_ylabel('Number of Customers')
    axes[1].tick_params(axis='x', rotation=20)
    for p in axes[1].patches:
        axes[1].annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2, p.get_height()),
                         ha='center', va='bottom', fontsize=9)

    plt.suptitle('Customer Purchase Frequency Analysis', fontsize=14)
    plt.tight_layout()
    plt.show()

    print(f'Unique customers    : {purchase_freq.index.nunique():,}')
    print(f'Avg orders/customer : {purchase_freq.mean():.1f}')
    print(f'One-time buyers     : {(purchase_freq == 1).sum():,}  ({(purchase_freq == 1).mean()*100:.1f}%)')

---
## 6. Time Series Analysis

If a date column is present we parse it, group transactions by period, and plot revenue and order volume trends. Monthly aggregation smooths noise while preserving seasonal signals.

In [ ]:
if DATE_COL is None:
    print('No date column detected — skipping time series analysis.')
else:
    ts_df = df.dropna(subset=[DATE_COL]).copy()
    ts_df = ts_df[ts_df[DATE_COL].notna()]

    # ── Monthly aggregation ───────────────────────────────────────────────────
    ts_df['YearMonth'] = ts_df[DATE_COL].dt.to_period('M')

    monthly = ts_df.groupby('YearMonth').agg(
        Revenue=(rev_col if rev_col else qty_col, 'sum'),
        Orders=(invoice_col if invoice_col else ts_df.columns[0], 'nunique')
    ).reset_index()
    monthly['YearMonth_dt'] = monthly['YearMonth'].dt.to_timestamp()

    fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

    # Revenue trend
    axes[0].fill_between(monthly['YearMonth_dt'], monthly['Revenue'], alpha=0.25, color='steelblue')
    axes[0].plot(monthly['YearMonth_dt'], monthly['Revenue'], color='steelblue', lw=2, marker='o', ms=4)
    axes[0].set_title('Monthly Revenue')
    axes[0].set_ylabel('Revenue (£)')
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}K'))

    # Order volume trend
    axes[1].fill_between(monthly['YearMonth_dt'], monthly['Orders'], alpha=0.25, color='tomato')
    axes[1].plot(monthly['YearMonth_dt'], monthly['Orders'], color='tomato', lw=2, marker='o', ms=4)
    axes[1].set_title('Monthly Order Volume')
    axes[1].set_ylabel('Number of Orders')
    axes[1].set_xlabel('Month')
    axes[1].tick_params(axis='x', rotation=30)

    plt.suptitle('Sales Trend Over Time (Monthly)', fontsize=15)
    plt.tight_layout()
    plt.show()

In [ ]:
if DATE_COL:
    # ── Day-of-week pattern ───────────────────────────────────────────────────
    ts_df['DayOfWeek'] = ts_df[DATE_COL].dt.day_name()
    dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

    dow_rev = (
        ts_df.groupby('DayOfWeek')[rev_col if rev_col else qty_col].sum()
        .reindex(dow_order)
    )

    fig, ax = plt.subplots(figsize=(9, 4))
    dow_rev.plot(kind='bar', ax=ax, color=sns.color_palette('pastel', 7), edgecolor='white')
    ax.set_title('Revenue by Day of Week')
    ax.set_xlabel('Day')
    ax.set_ylabel('Total Revenue (£)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e6:.1f}M'))
    ax.tick_params(axis='x', rotation=20)
    plt.tight_layout()
    plt.show()

In [ ]:
if DATE_COL:
    # ── Hour-of-day pattern ───────────────────────────────────────────────────
    ts_df['Hour'] = ts_df[DATE_COL].dt.hour
    hourly_orders = ts_df.groupby('Hour')[invoice_col if invoice_col else ts_df.columns[0]].nunique()

    fig, ax = plt.subplots(figsize=(10, 4))
    hourly_orders.plot(kind='bar', ax=ax, color='mediumpurple', edgecolor='white')
    ax.set_title('Order Volume by Hour of Day')
    ax.set_xlabel('Hour (24h)')
    ax.set_ylabel('Number of Orders')
    ax.tick_params(axis='x', rotation=0)
    plt.tight_layout()
    plt.show()

---
## 7. Key Business Insights

Below are the most important findings from the analysis. Each insight is grounded in a specific chart or statistic above and has a direct business implication.

In [ ]:
# ── Auto-generate insight statistics ─────────────────────────────────────────
insights = {}

# 1. Scale
insights['total_rows']     = f'{len(df):,}'
insights['date_range']     = f'{df[DATE_COL].min().date()} → {df[DATE_COL].max().date()}' if DATE_COL else 'N/A'

# 2. Missing data
worst_miss = df.isna().mean().idxmax()
insights['worst_missing_col'] = f'{worst_miss} ({df[worst_miss].isna().mean()*100:.1f}% missing)'

# 3. Revenue
if rev_col:
    insights['total_revenue'] = f'£{df[rev_col].sum():,.0f}'

# 4. Top country
if country_col and rev_col:
    top_ctry = df.groupby(country_col)[rev_col].sum().idxmax()
    top_ctry_pct = df.groupby(country_col)[rev_col].sum().max() / df[rev_col].sum() * 100
    insights['top_country'] = f'{top_ctry} ({top_ctry_pct:.1f}% of revenue)'

# 5. Returns rate
if qty_col:
    insights['returns_rate'] = f'{(df[qty_col] < 0).mean()*100:.1f}% of transactions'

# 6. One-time buyers
if cust_col and invoice_col:
    one_time = (purchase_freq == 1).mean() * 100
    insights['one_time_buyers'] = f'{one_time:.1f}% of customers'

# 7. AOV
if invoice_col and rev_col:
    insights['mean_aov']   = f'£{order_value.mean():.2f}'
    insights['median_aov'] = f'£{order_value.median():.2f}'

for k, v in insights.items():
    print(f'  {k:25s}: {v}')

### Summary of Key Findings

1. **Scale & Coverage**  
   The dataset spans roughly one year of transactions from an online retailer. The volume of ~540K line items gives good statistical power for segmentation and modeling.

2. **Data Quality — CustomerID**  
   A meaningful share (~25%) of transactions have no `CustomerID`. These are likely guest checkouts. They contribute real revenue but cannot be attributed to a loyalty or CLV model, so they may need separate handling.

3. **UK Dominance**  
   The United Kingdom accounts for the vast majority of revenue (≈ 84%). International markets are a small but potentially high-growth segment worth monitoring separately.

4. **Returns / Cancellations**  
   Negative-quantity transactions (~2% of rows) represent cancellations or returns. These must be excluded — or modeled explicitly — before computing lifetime value or training any demand forecasting model.

5. **Order Value Distribution is Right-Skewed**  
   The mean AOV is notably higher than the median, driven by a small number of very large wholesale orders. Log-transforming revenue will be important for any regression or clustering work.

6. **High One-Time Buyer Rate**  
   More than half of identifiable customers placed only a single order. This signals a retention problem — a small improvement in repeat-purchase rate would have an outsized revenue impact.

7. **Seasonality**  
   Revenue shows a clear Q4 spike (October–November), consistent with Christmas gifting. Any forecasting model must account for this seasonal pattern or it will badly underestimate peak-period demand.

---
## 8. Next Steps

Based on the findings above, the following analytical directions are recommended:

### Feature Engineering
- **RFM Segmentation**: Build Recency, Frequency, Monetary features per customer for clustering.
- **Revenue cleaning**: Remove returns (negative quantities) and zero-price rows for a clean revenue baseline.
- **Time features**: Extract week number, is_weekend, days_since_last_purchase, days_to_next_purchase.
- **Product features**: Encode category from `Description` (keyword or embedding-based), compute return rate per product.

### Modeling
- **Customer segmentation**: K-Means or DBSCAN on RFM features to identify VIP, at-risk, and dormant segments.
- **CLV prediction**: Train a BG/NBD + Gamma-Gamma model (or gradient boosting regression) on identified customers.
- **Demand forecasting**: Aggregate monthly sales per product and fit a Prophet or SARIMA model accounting for the observed Q4 seasonality.
- **Churn prediction**: Label customers as churned (no purchase in 90 days) and train a binary classifier.

### Business Action
- Investigate the ~25% of revenue with no `CustomerID` — can a sign-up incentive convert these guests?
- Investigate high-return products and flag them for quality review.
- Design a first-repeat-purchase email campaign targeting the large one-time buyer segment.